# Targeted Gene Expression ROC/AUC Analysis
### Esophageal Squamous Cell Carcinoma (tumor vs adjacent normal)
**Focus:** ROC & AUC for a predefined panel of 13 mRNAs and 4 lcnRNAs

---
**Pipeline:**
1. Install & import
2. Target gene list
3. Download GEO datasets
4. Inspect dataset
5. Extract phenotype labels
6. Build & preprocess expression matrix
7. GPL annotation → probe–gene mapping
8. Match target genes → probes
9. ROC/AUC per target gene
10. Individual ROC curve grid



## 1. Import

In [ ]:
import os, warnings, re
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import GEOparse

from scipy import stats
from sklearn.metrics import roc_curve, auc
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ── Shared style ──
DPI     = 500
PALETTE = {"disease": "#E63946", "control": "#457B9D"}

plt.rcParams.update({
    "figure.dpi":        150,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":         11,
})

OUTDIR = "./results"
os.makedirs(OUTDIR, exist_ok=True)

## 2. Target Gene List

In [ ]:
TARGET_GENES = [
    "BCOR", "TET2", "MAP3K1", "KMT2A", "NEAT1", "KCNQ1OT1",
    "DDX6", "TAS2R30", "RNU4-1", "OSTCP8", "NRTN", "LCMT1-AS2",
    "LAMA5-AS1", "HOOK3", "SP1", "KMT2D", "KMT2C"
]

## 3. Download GEO datasets

In [ ]:
DESTDIR = "./geo_data"
os.makedirs(DESTDIR, exist_ok=True)

try:
    gse = GEOparse.get_GEO("GSE20347", destdir=DESTDIR, silent=False)

except Exception as e:
    raise

## 4. Inspect Dataset

In [ ]:
for k in ["Series_title", "Series_summary", "Series_platform_id"]:
    val = gse.metadata.get(k, ["N/A"])
    val = val if isinstance(val, list) else [val]
    print(f"\n{k}:")
    for v in val[:2]:
        print(f"  {v[:300]}")

print(f"\nTotal samples : {len(gse.gsms)}")
print(f"Platform(s)   : {list(gse.gpls.keys())}")

pheno = gse.phenotype_data.copy()
print("\nPhenotype columns:")
for col in pheno.columns:
    print(f"  [{col}]: {pheno[col].unique()[:3]}")

## 5. Extract Phenotype Labels

In [ ]:
CONTROL_KW = ["normal","control","healthy","adjacent","non-tumor","para",
              "benign","noncancer","non-cancer","uninvolved","donor","matched"]
DISEASE_KW = ["tumor","cancer","carcinoma","adenocarcinoma","malignant",
              "squamous","sarcoma","leukemia","lymphoma","melanoma",
              "glioma","hepato","gastric","affected"]

condition_col = None
best_score    = 0
for col in pheno.columns:
    vals_str = " ".join(str(v).lower() for v in pheno[col].dropna().unique())
    score = (sum(kw in vals_str for kw in DISEASE_KW) +
             sum(kw in vals_str for kw in CONTROL_KW))
    if score > best_score and 2 <= len(pheno[col].dropna().unique()) <= 10:
        best_score    = score
        condition_col = col

print(f"✓ Auto-detected column: '{condition_col}'")
print(f"  Values: {pheno[condition_col].unique()}")
CONDITION_COL = condition_col   # ← override manually if wrong

In [ ]:
def map_label(val):
    v = str(val).lower()
    if any(kw in v for kw in CONTROL_KW):   # control FIRST
        return 0
    elif any(kw in v for kw in DISEASE_KW):
        return 1
    return np.nan

labels = pheno[CONDITION_COL].map(map_label)

print("Label mapping:")
for val in pheno[CONDITION_COL].unique():
    mapped = map_label(val)
    sym = {0: "→ 0 (Control)", 1: "→ 1 (Disease)"}.get(mapped, "→ ⚠ UNMAPPED")
    print(f"  {sym}  |  {val}")

labels = labels.dropna().astype(int)
print(f"\n0 = Control : {(labels==0).sum()} samples")
print(f"1 = Disease : {(labels==1).sum()} samples")
print(f"Total       : {len(labels)} samples")

## 6. Build & Preprocess Expression Matrix

In [ ]:
expr_data = {}
for gsm_name, gsm in gse.gsms.items():
    if gsm_name in labels.index:
        try:
            s = gsm.table.set_index("ID_REF")["VALUE"].astype(float)
            expr_data[gsm_name] = s
        except Exception as e:
            print(f"  ⚠ Skipped {gsm_name}: {e}")

expr   = pd.DataFrame(expr_data)
common = [s for s in expr.columns if s in labels.index]
expr   = expr[common]
labels = labels.loc[common]

expr.index   = expr.index.astype(str)
labels.index = labels.index.astype(str)
print(f"✓ Raw matrix : {expr.shape[0]} probes × {expr.shape[1]} samples")

In [ ]:
# NA filter → log2 → quantile normalisation
na_frac = expr.isnull().mean(axis=1)
expr    = expr.loc[na_frac <= 0.20]
expr    = expr.apply(lambda row: row.fillna(row.median()), axis=1)
print(f"After NA filter    : {expr.shape[0]} probes")

min_val = expr.min().min()
expr    = np.log2(expr - min_val + 1) if min_val <= 0 else np.log2(expr + 1)
print("Log2 transform     : done")

def quantile_normalize(df):
    rank_mean = df.stack().groupby(df.rank(method="first").stack().astype(int)).mean()
    return df.rank(method="min").stack().astype(int).map(rank_mean).unstack()

expr = quantile_normalize(expr)
print("Quantile normalize : done")
print(f"Final matrix       : {expr.shape[0]} probes × {expr.shape[1]} samples")

## 7. GPL Annotation → Probe–Gene Mapping

In [ ]:
gpl_id = list(gse.gpls.keys())[0]
gpl    = gse.gpls[gpl_id]
print(f"Platform : {gpl_id}")
print(f"Columns  : {gpl.table.columns.tolist()}")
gpl.table.head(4)

In [ ]:
GENE_COLS = ["Gene Symbol","gene_symbol","GENE_SYMBOL","Symbol","symbol",
             "Gene_Symbol","GB_ACC","GenBank Accession","gene_assignment",
             "ILMN_Gene","Entrez_Gene_ID","Description"]

name_col = None
for pref in GENE_COLS:
    if pref in gpl.table.columns:
        sample = gpl.table[pref].dropna().astype(str)
        if sample.str.match(r'^[A-Za-z]').sum() > 20:
            name_col = pref
            print(f"✓ Gene column: '{name_col}'")
            break

if name_col is None:
    print("⚠ Could not auto-detect gene column.")
    name_col = gpl.table.columns[1]
    print(f"Defaulting to: '{name_col}'")

id_col = "ID" if "ID" in gpl.table.columns else gpl.table.columns[0]
anno   = gpl.table[[id_col, name_col]].copy()
anno.columns = ["probe_id", "gene_raw"]
anno["probe_id"]   = anno["probe_id"].astype(str).str.strip()
anno["gene_clean"] = (anno["gene_raw"].astype(str)
                          .str.strip().str.upper()
                          .str.split(r"\s*[/|;,]\s*", regex=True).str[0]
                          .str.strip())

anno = anno[anno["probe_id"].isin(expr.index)].copy()
anno = anno[anno["gene_clean"].notna() & (anno["gene_clean"] != "") & (anno["gene_clean"] != "NAN")]
print(f"✓ Annotation rows matching expr : {len(anno)}")
print(f"  Unique genes annotated        : {anno['gene_clean'].nunique()}")

## 8. Match Target Genes → Probes

In [ ]:
anno_grouped = anno.groupby("gene_clean")["probe_id"].apply(list).to_dict()

matched   = {}
unmatched = []

for gene in TARGET_GENES:
    g      = gene.upper().strip()
    probes = [p for p in anno_grouped.get(g, []) if p in expr.index]
    if probes:
        matched[gene] = probes
    else:
        unmatched.append(gene)

print(f"✓ Matched   : {len(matched)} / {len(TARGET_GENES)}")
print(f"✗ Unmatched : {len(unmatched)}")
if unmatched:
    for g in unmatched: print(f"   {g}")

## 9. ROC/AUC per Target Gene

In [ ]:
expr.index   = expr.index.astype(str)
labels.index = labels.index.astype(str)
y = labels.values

roc_results = []
for gene, probes in matched.items():
    if len(probes) == 1:
        x = expr.loc[probes[0], labels.index].values.astype(float)
    else:
        sub  = expr.loc[probes, labels.index]
        best = sub.var(axis=1).idxmax()
        x    = sub.loc[best].values.astype(float)

    fpr, tpr, _ = roc_curve(y, x)
    roc_auc     = auc(fpr, tpr)
    flipped = False
    if roc_auc < 0.5:
        fpr, tpr, _ = roc_curve(y, -x)
        roc_auc     = auc(fpr, tpr)
        flipped     = True

    log2fc = x[y==1].mean() - x[y==0].mean()
    _, pval = stats.ttest_ind(x[y==1], x[y==0], equal_var=False)

    roc_results.append({
        "Gene":      gene,
        "probes":    ", ".join(probes[:3]),
        "n_probes":  len(probes),
        "AUC":       roc_auc,
        "log2FC":    log2fc,
        "p_value":   pval,
        "direction": "Up" if log2fc > 0 else "Down",
        "flipped":   flipped,
        "fpr":       fpr,
        "tpr":       tpr,
    })

results_df = pd.DataFrame(
    [{k:v for k,v in r.items() if k not in ["fpr","tpr"]} for r in roc_results]
).sort_values("AUC", ascending=False).reset_index(drop=True)

print(f"✓ ROC/AUC computed for {len(results_df)} genes\n")
results_df[["Gene","AUC","log2FC","p_value","direction","n_probes"]]

## 10. Individual ROC Curve Grid

In [ ]:
n_genes = len(roc_results)
n_cols  = 4
n_rows  = int(np.ceil(n_genes / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 3.2))
axes = axes.flatten()

for i, res in enumerate(sorted(roc_results, key=lambda r: r["AUC"], reverse=True)):
    ax    = axes[i]
    color = PALETTE["disease"] if res["log2FC"] > 0 else PALETTE["control"]
    ax.plot(res["fpr"], res["tpr"], lw=2, color=color)
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.4)
    ax.set_title(res["Gene"], fontsize=10, fontweight="bold")
    ax.text(0.97, 0.07, f"AUC={res['AUC']:.3f}", transform=ax.transAxes,
            ha="right", fontsize=9, color=color,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.8))
    ax.set_xlabel("False Positive Rate", fontsize=8)
    ax.set_ylabel("True Positive Rate", fontsize=8)
    ax.set_xticks([0, 0.5, 1]); ax.set_yticks([0, 0.5, 1])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("ROC Curves — Gene Panel\n (Esophageal Squamous Cell Carcinoma)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
for ext in ("png", "tiff"):
    plt.savefig(f"{OUTDIR}/ROC_grid.{ext}", dpi=DPI, bbox_inches="tight")
plt.show()
print("✓ Saved: ROC_grid .png / .tiff")

In [ ]:
for res in sorted(roc_results, key=lambda r: r["AUC"], reverse=True):
    fig, ax = plt.subplots(figsize=(4, 4))
    color = PALETTE["disease"] if res["log2FC"] > 0 else PALETTE["control"]
    ax.plot(res["fpr"], res["tpr"], lw=2, color=color)
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.4)
    ax.set_title(res["Gene"], fontsize=10, fontweight="bold")
    ax.text(0.97, 0.07, f"AUC={res['AUC']:.3f}", transform=ax.transAxes,
            ha="right", fontsize=9, color=color,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.8))
    ax.set_xlabel("False Positive Rate", fontsize=8)
    ax.set_ylabel("True Positive Rate", fontsize=8)
    ax.set_xticks([0, 0.5, 1]); ax.set_yticks([0, 0.5, 1])
    plt.tight_layout()
    safe_name = res["Gene"].replace("/", "_")
    for ext in ("png", "tiff"):
        plt.savefig(f"{OUTDIR}/ROC_{safe_name}.{ext}", dpi=DPI, bbox_inches="tight")
    plt.close()

print(f"✓ Saved {len(roc_results)} individual ROC figures to {OUTDIR}")